# Compare Two Submissions (Diff, Class Changes, Plots)

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, os, time, pathlib
plt.style.use('seaborn-v0_8')
A = 'outputs/submissions/20251110-110523_tf_efficientnet_b7_ns_528px_rot90_jitter_affine_rrc_mixup_only_ls0.05_cosine_f1_0.9910_result(0.0000)_effb7_528_tta90_hflip_dn15_nocache.csv'
B = 'outputs/submissions/20251110-113208_tf_efficientnet_b7_ns_528px_rot90_jitter_affine_rrc_mixup_only_ls0.05_cosine_f1_0.9910_result(0.0000)_effb7_528_tta90_hflip_swinirproxy.csv'
out_dir = pathlib.Path('reports/summary') / time.strftime('%Y%m%d-%H%M%S')
out_dir.mkdir(parents=True, exist_ok=True)
sa = pd.read_csv(A)
sb = pd.read_csv(B)
assert (sa['ID'] == sb['ID']).all(), 'Submission ID orders differ'
ida = sa['ID'].values
ya = sa['target'].values
yb = sb['target'].values
agree = (ya == yb)
agree_rate = float(agree.mean())
print('Agreement:', agree_rate)
# Per-class counts
num_classes = max(int(ya.max()), int(yb.max())) + 1
cnt_a = np.bincount(ya, minlength=num_classes)
cnt_b = np.bincount(yb, minlength=num_classes)
df_cnt = pd.DataFrame({'class': range(num_classes), 'A_count': cnt_a, 'B_count': cnt_b})
ax = df_cnt.set_index('class').plot(kind='bar', figsize=(10,4), title='Predicted counts per class (A vs B)')
plt.tight_layout(); plt.savefig(out_dir/'pred_counts_ab.png', dpi=150); plt.close()
# Change matrix
cm = np.zeros((num_classes, num_classes), dtype=int)
for c_from, c_to in zip(ya, yb): cm[c_from, c_to] += 1
plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.title('Changes A→B (counts)'); plt.xlabel('B'); plt.ylabel('A')
plt.tight_layout(); plt.savefig(out_dir/'change_matrix_A_to_B.png', dpi=150); plt.close()
# Changed samples per class (A correct? we don't have labels; show changed count by from/to)
changed = (ya != yb)
chg_from = np.bincount(ya[changed], minlength=num_classes)
chg_to = np.bincount(yb[changed], minlength=num_classes)
df_chg = pd.DataFrame({'class': range(num_classes), 'changed_from': chg_from, 'changed_to': chg_to})
ax = df_chg.set_index('class').plot(kind='bar', figsize=(10,4), title='Changed predictions by class')
plt.tight_layout(); plt.savefig(out_dir/'changed_by_class.png', dpi=150); plt.close()
print('Saved figures to', out_dir)
